# 01 - Carga y Limpieza de Datos
## Prueba Técnica - Científico de Datos Senior
### Gestión de Costos Operativos en un Proyecto de Construcción

---

**Objetivo:** Cargar los datos crudos desde S3, identificar y corregir problemas de calidad, y generar un dataset limpio y consolidado listo para el análisis exploratorio.

**Fuentes de datos:**
- `X.csv` — Serie de precios de materia prima X
- `Y.csv` — Serie de precios de materia prima Y
- `Z.csv` — Serie de precios de materia prima Z
- `historico_equipos.csv` — Precios consolidados de materias primas y equipos

## 1. Configuración del entorno

In [1]:
import boto3
import pandas as pd
import numpy as np
from io import StringIO
import warnings
warnings.filterwarnings('ignore')

# Configuración S3
BUCKET = 'consulting-dataknow-prueba-tecnica'
PREFIX_CRUDO = 'datos_crudos/'
PREFIX_PROCESADO = 'datos_procesados/'

s3 = boto3.client('s3')

print('Entorno configurado correctamente.')

Entorno configurado correctamente.


## 2. Funciones auxiliares

In [2]:
def leer_csv_s3(bucket, key, **kwargs):
    """Lee un CSV desde S3 y lo devuelve como DataFrame."""
    obj = s3.get_object(Bucket=bucket, Key=key)
    contenido = obj['Body'].read().decode('utf-8-sig')
    return pd.read_csv(StringIO(contenido), **kwargs)


def guardar_csv_s3(df, bucket, key):
    """Guarda un DataFrame como CSV en S3."""
    buffer = StringIO()
    df.to_csv(buffer, index=False)
    s3.put_object(Bucket=bucket, Key=key, Body=buffer.getvalue())
    print(f'Archivo guardado en s3://{bucket}/{key}')


def diagnostico(df, nombre):
    """Imprime un diagnóstico completo de un DataFrame."""
    print(f'\n{"=" * 60}')
    print(f'  DIAGNÓSTICO: {nombre}')
    print(f'{"=" * 60}')
    print(f'  Shape: {df.shape[0]} filas x {df.shape[1]} columnas')
    print(f'  Columnas: {df.columns.tolist()}')
    print(f'  Tipos de dato:')
    for col in df.columns:
        print(f'    {col}: {df[col].dtype}')
    print(f'  Valores nulos:')
    for col in df.columns:
        nulos = df[col].isnull().sum()
        print(f'    {col}: {nulos} ({nulos/len(df)*100:.2f}%)')
    print(f'  Filas duplicadas: {df.duplicated().sum()}')
    if 'Date' in df.columns:
        print(f'  Fechas duplicadas: {df["Date"].duplicated().sum()}')
        print(f'  Rango de fechas: {df["Date"].min()} → {df["Date"].max()}')
    print()

## 3. Verificación de archivos en S3

In [3]:
response = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX_CRUDO)

print('Archivos disponibles en S3:')
print('-' * 50)
for obj in response.get('Contents', []):
    if obj['Size'] > 0:  # Ignorar el directorio vacío
        print(f"  {obj['Key']:45s} {obj['Size']:>10,} bytes")

Archivos disponibles en S3:
--------------------------------------------------
  datos_crudos/X.csv                               164,774 bytes
  datos_crudos/Y.csv                                73,199 bytes
  datos_crudos/Z.csv                                69,831 bytes
  datos_crudos/historico_equipos.csv               159,264 bytes


## 4. Carga de datos crudos

Cargamos cada archivo identificando sus particularidades de formato.

### 4.1 Materia Prima X

In [4]:
# X.csv: formato estándar CSV, separador coma, fechas YYYY-MM-DD
df_x = leer_csv_s3(BUCKET, f'{PREFIX_CRUDO}X.csv')

print('Vista previa de X.csv (datos crudos):')
print(df_x.head())
diagnostico(df_x, 'X.csv - Crudo')

Vista previa de X.csv (datos crudos):
         Date  Price
0  2024-04-04  89.18
1  2024-04-03  89.35
2  2024-04-02  88.92
3  2024-04-01  87.42
4  2024-03-28  87.48

  DIAGNÓSTICO: X.csv - Crudo
  Shape: 9144 filas x 2 columnas
  Columnas: ['Date', 'Price']
  Tipos de dato:
    Date: object
    Price: float64
  Valores nulos:
    Date: 0 (0.00%)
    Price: 0 (0.00%)
  Filas duplicadas: 0
  Fechas duplicadas: 0
  Rango de fechas: 1988-06-27 → 2024-04-04



### 4.2 Materia Prima Y

In [5]:
# Y.csv: PROBLEMAS IDENTIFICADOS:
# 1. Separador: punto y coma (;) en vez de coma
# 2. Decimales: coma europea (547,33) en vez de punto
# 3. Fechas: formato DD/MM/YYYY en vez de YYYY-MM-DD
# 4. BOM: carácter invisible UTF-8 BOM al inicio del archivo

df_y = leer_csv_s3(BUCKET, f'{PREFIX_CRUDO}Y.csv', sep=';', decimal=',')

print('Vista previa de Y.csv (datos crudos):')
print(df_y.head())
diagnostico(df_y, 'Y.csv - Crudo')

Vista previa de Y.csv (datos crudos):
        Date   Price
0  12/9/2023  547.33
1  11/9/2023  546.00
2   8/9/2023  545.00
3   7/9/2023  550.00
4   6/9/2023  552.50

  DIAGNÓSTICO: Y.csv - Crudo
  Shape: 4485 filas x 2 columnas
  Columnas: ['Date', 'Price']
  Tipos de dato:
    Date: object
    Price: float64
  Valores nulos:
    Date: 0 (0.00%)
    Price: 0 (0.00%)
  Filas duplicadas: 0
  Fechas duplicadas: 0
  Rango de fechas: 1/1/2007 → 9/9/2022



### 4.3 Materia Prima Z

In [6]:
# Z.csv: PROBLEMA IDENTIFICADO:
# 1. Columnas invertidas: Price,Date en vez de Date,Price

df_z = leer_csv_s3(BUCKET, f'{PREFIX_CRUDO}Z.csv')

print('Vista previa de Z.csv (datos crudos):')
print(f'Columnas originales: {df_z.columns.tolist()}')
print(df_z.head())
diagnostico(df_z, 'Z.csv - Crudo')

Vista previa de Z.csv (datos crudos):
Columnas originales: ['Price', 'Date']
     Price        Date
0  2225.25  2010-01-01
1  2225.25  2010-01-04
2  2246.50  2010-01-05
3  2302.50  2010-01-06
4  2306.50  2010-01-07

  DIAGNÓSTICO: Z.csv - Crudo
  Shape: 3565 filas x 2 columnas
  Columnas: ['Price', 'Date']
  Tipos de dato:
    Price: float64
    Date: object
  Valores nulos:
    Price: 0 (0.00%)
    Date: 0 (0.00%)
  Filas duplicadas: 0
  Fechas duplicadas: 0
  Rango de fechas: 2010-01-01 → 2023-08-31



### 4.4 Histórico de Equipos

In [7]:
# historico_equipos.csv: formato limpio, sin problemas aparentes
df_equipos = leer_csv_s3(BUCKET, f'{PREFIX_CRUDO}historico_equipos.csv')

print('Vista previa de historico_equipos.csv (datos crudos):')
print(df_equipos.head())
diagnostico(df_equipos, 'historico_equipos.csv - Crudo')

Vista previa de historico_equipos.csv (datos crudos):
         Date  Price_X  Price_Y  Price_Z  Price_Equipo1  Price_Equipo2
0  2010-01-04    80.12    527.5  2225.25         434.73         931.73
1  2010-01-05    80.59    527.5  2246.50         449.97         968.56
2  2010-01-06    81.89    527.5  2302.50         444.48         960.51
3  2010-01-07    81.51    527.5  2306.50         440.90         960.14
4  2010-01-08    81.37    552.5  2261.25         448.82         949.55

  DIAGNÓSTICO: historico_equipos.csv - Crudo
  Shape: 3530 filas x 6 columnas
  Columnas: ['Date', 'Price_X', 'Price_Y', 'Price_Z', 'Price_Equipo1', 'Price_Equipo2']
  Tipos de dato:
    Date: object
    Price_X: float64
    Price_Y: float64
    Price_Z: float64
    Price_Equipo1: float64
    Price_Equipo2: float64
  Valores nulos:
    Date: 0 (0.00%)
    Price_X: 0 (0.00%)
    Price_Y: 0 (0.00%)
    Price_Z: 0 (0.00%)
    Price_Equipo1: 0 (0.00%)
    Price_Equipo2: 0 (0.00%)
  Filas duplicadas: 0
  Fechas duplica

## 5. Limpieza de datos

### Problemas detectados y acciones:

| Archivo | Problema | Acción |
|---------|----------|--------|
| Y.csv | Separador `;`, decimal `,` | Corregido en la carga con `sep=';', decimal=','` |
| Y.csv | BOM UTF-8 al inicio | Corregido con `encoding='utf-8-sig'` |
| Y.csv | Fechas en formato DD/MM/YYYY | Convertir con `dayfirst=True` |
| Z.csv | Columnas invertidas (Price, Date) | Reordenar columnas |
| X.csv | Orden descendente (más reciente primero) | Ordenar ascendente |
| Y.csv | Orden descendente (más reciente primero) | Ordenar ascendente |
| Todos | Fechas como string | Convertir a datetime |

### 5.1 Limpieza de X

In [8]:
# Convertir fecha a datetime
df_x['Date'] = pd.to_datetime(df_x['Date'])

# Ordenar de más antiguo a más reciente
df_x = df_x.sort_values('Date').reset_index(drop=True)

diagnostico(df_x, 'X - Limpio')
print(df_x.head())


  DIAGNÓSTICO: X - Limpio
  Shape: 9144 filas x 2 columnas
  Columnas: ['Date', 'Price']
  Tipos de dato:
    Date: datetime64[ns]
    Price: float64
  Valores nulos:
    Date: 0 (0.00%)
    Price: 0 (0.00%)
  Filas duplicadas: 0
  Fechas duplicadas: 0
  Rango de fechas: 1988-06-27 00:00:00 → 2024-04-04 00:00:00

        Date  Price
0 1988-06-27  15.10
1 1988-06-28  15.27
2 1988-06-29  15.47
3 1988-06-30  14.85
4 1988-07-01  14.60


### 5.2 Limpieza de Y

In [9]:
# Convertir fecha a datetime con formato europeo (día primero)
df_y['Date'] = pd.to_datetime(df_y['Date'], dayfirst=True)

# Ordenar de más antiguo a más reciente
df_y = df_y.sort_values('Date').reset_index(drop=True)

diagnostico(df_y, 'Y - Limpio')
print(df_y.head())


  DIAGNÓSTICO: Y - Limpio
  Shape: 4485 filas x 2 columnas
  Columnas: ['Date', 'Price']
  Tipos de dato:
    Date: datetime64[ns]
    Price: float64
  Valores nulos:
    Date: 0 (0.00%)
    Price: 0 (0.00%)
  Filas duplicadas: 0
  Fechas duplicadas: 0
  Rango de fechas: 2006-07-11 00:00:00 → 2023-09-12 00:00:00

        Date  Price
0 2006-07-11  555.0
1 2006-07-12  555.0
2 2006-07-13  555.0
3 2006-07-14  555.0
4 2006-07-17  555.0


### 5.3 Limpieza de Z

In [10]:
# Reordenar columnas: Date primero, Price después
df_z = df_z[['Date', 'Price']]

# Convertir fecha a datetime
df_z['Date'] = pd.to_datetime(df_z['Date'])

# Ordenar por fecha
df_z = df_z.sort_values('Date').reset_index(drop=True)

diagnostico(df_z, 'Z - Limpio')
print(df_z.head())


  DIAGNÓSTICO: Z - Limpio
  Shape: 3565 filas x 2 columnas
  Columnas: ['Date', 'Price']
  Tipos de dato:
    Date: datetime64[ns]
    Price: float64
  Valores nulos:
    Date: 0 (0.00%)
    Price: 0 (0.00%)
  Filas duplicadas: 0
  Fechas duplicadas: 0
  Rango de fechas: 2010-01-01 00:00:00 → 2023-08-31 00:00:00

        Date    Price
0 2010-01-01  2225.25
1 2010-01-04  2225.25
2 2010-01-05  2246.50
3 2010-01-06  2302.50
4 2010-01-07  2306.50


### 5.4 Limpieza de Histórico de Equipos

In [11]:
# Convertir fecha a datetime
df_equipos['Date'] = pd.to_datetime(df_equipos['Date'])

# Ordenar por fecha
df_equipos = df_equipos.sort_values('Date').reset_index(drop=True)

diagnostico(df_equipos, 'Histórico Equipos - Limpio')
print(df_equipos.head())


  DIAGNÓSTICO: Histórico Equipos - Limpio
  Shape: 3530 filas x 6 columnas
  Columnas: ['Date', 'Price_X', 'Price_Y', 'Price_Z', 'Price_Equipo1', 'Price_Equipo2']
  Tipos de dato:
    Date: datetime64[ns]
    Price_X: float64
    Price_Y: float64
    Price_Z: float64
    Price_Equipo1: float64
    Price_Equipo2: float64
  Valores nulos:
    Date: 0 (0.00%)
    Price_X: 0 (0.00%)
    Price_Y: 0 (0.00%)
    Price_Z: 0 (0.00%)
    Price_Equipo1: 0 (0.00%)
    Price_Equipo2: 0 (0.00%)
  Filas duplicadas: 0
  Fechas duplicadas: 0
  Rango de fechas: 2010-01-04 00:00:00 → 2023-08-31 00:00:00

        Date  Price_X  Price_Y  Price_Z  Price_Equipo1  Price_Equipo2
0 2010-01-04    80.12    527.5  2225.25         434.73         931.73
1 2010-01-05    80.59    527.5  2246.50         449.97         968.56
2 2010-01-06    81.89    527.5  2302.50         444.48         960.51
3 2010-01-07    81.51    527.5  2306.50         440.90         960.14
4 2010-01-08    81.37    552.5  2261.25         448.82  

## 6. Validación cruzada

Verificamos que los datos individuales de X, Y, Z sean consistentes con el histórico consolidado.

In [12]:
# Comparar rangos de fechas
print('RANGOS DE FECHAS')
print('=' * 60)
print(f'  X:       {df_x["Date"].min().date()} → {df_x["Date"].max().date()}  ({len(df_x):,} registros)')
print(f'  Y:       {df_y["Date"].min().date()} → {df_y["Date"].max().date()}  ({len(df_y):,} registros)')
print(f'  Z:       {df_z["Date"].min().date()} → {df_z["Date"].max().date()}  ({len(df_z):,} registros)')
print(f'  Equipos: {df_equipos["Date"].min().date()} → {df_equipos["Date"].max().date()}  ({len(df_equipos):,} registros)')

# Datos posteriores al histórico de equipos (útiles para proyección)
fecha_corte = df_equipos['Date'].max()
print(f'\nDATOS POSTERIORES AL HISTÓRICO (después de {fecha_corte.date()}):')
print(f'  X: {len(df_x[df_x["Date"] > fecha_corte]):,} registros hasta {df_x["Date"].max().date()}')
print(f'  Y: {len(df_y[df_y["Date"] > fecha_corte]):,} registros hasta {df_y["Date"].max().date()}')
print(f'  Z: {len(df_z[df_z["Date"] > fecha_corte]):,} registros hasta {df_z["Date"].max().date()}')

RANGOS DE FECHAS
  X:       1988-06-27 → 2024-04-04  (9,144 registros)
  Y:       2006-07-11 → 2023-09-12  (4,485 registros)
  Z:       2010-01-01 → 2023-08-31  (3,565 registros)
  Equipos: 2010-01-04 → 2023-08-31  (3,530 registros)

DATOS POSTERIORES AL HISTÓRICO (después de 2023-08-31):
  X: 154 registros hasta 2024-04-04
  Y: 8 registros hasta 2023-09-12
  Z: 0 registros hasta 2023-08-31


In [14]:
# Verificar consistencia de precios entre CSVs individuales y el histórico
print('VERIFICACIÓN DE CONSISTENCIA')
print('=' * 60)

# Merge X con equipos por fecha
check_x = pd.merge(df_equipos[['Date', 'Price_X']], df_x, on='Date', how='inner')
diff_x = (check_x['Price_X'] - check_x['Price']).abs()
print(f'  X: {len(check_x):,} fechas en común, diferencia máxima: {diff_x.max():.4f}')

# Merge Y con equipos por fecha
check_y = pd.merge(df_equipos[['Date', 'Price_Y']], df_y, on='Date', how='inner')
diff_y = (check_y['Price_Y'] - check_y['Price']).abs()
print(f'  Y: {len(check_y):,} fechas en común, diferencia máxima: {diff_y.max():.4f}')

# Merge Z con equipos por fecha
check_z = pd.merge(df_equipos[['Date', 'Price_Z']], df_z, on='Date', how='inner')
diff_z = (check_z['Price_Z'] - check_z['Price']).abs()
print(f'  Z: {len(check_z):,} fechas en común, diferencia máxima: {diff_z.max():.4f}')

if diff_x.max() < 0.01 and diff_y.max() < 0.01 and diff_z.max() < 0.01:
    print('\nLos datos individuales son consistentes con el histórico consolidado.')
else:
    print('\nSe detectaron diferencias entre los datos individuales y el histórico.')

VERIFICACIÓN DE CONSISTENCIA
  X: 3,530 fechas en común, diferencia máxima: 0.0000
  Y: 3,530 fechas en común, diferencia máxima: 0.0000
  Z: 3,530 fechas en común, diferencia máxima: 0.0000

Los datos individuales son consistentes con el histórico consolidado.


## 7. Resumen de la limpieza

In [15]:
print('RESUMEN DE LIMPIEZA')
print('=' * 60)
print()
print('Problemas detectados y corregidos:')
print('-' * 60)
print('  Y.csv:')
print('     Separador cambiado de ";" a ","')
print('     Decimales convertidos de coma europea a punto')
print('     Fechas convertidas de DD/MM/YYYY a datetime')
print('     BOM UTF-8 removido')
print('  Z.csv:')
print('     Columnas reordenadas de (Price, Date) a (Date, Price)')
print('  Todos los archivos:')
print('     Fechas convertidas a datetime')
print('     Ordenados cronológicamente (ascendente)')
print('     Sin valores nulos')
print('     Sin duplicados')
print('     Consistencia verificada entre CSVs individuales y consolidado')
print()
print('Datasets resultantes:')
print('-' * 60)
print(f'  df_x:       {df_x.shape} — Materia prima X')
print(f'  df_y:       {df_y.shape} — Materia prima Y')
print(f'  df_z:       {df_z.shape} — Materia prima Z')
print(f'  df_equipos: {df_equipos.shape} — Histórico consolidado')

RESUMEN DE LIMPIEZA

Problemas detectados y corregidos:
------------------------------------------------------------
  Y.csv:
     Separador cambiado de ";" a ","
     Decimales convertidos de coma europea a punto
     Fechas convertidas de DD/MM/YYYY a datetime
     BOM UTF-8 removido
  Z.csv:
     Columnas reordenadas de (Price, Date) a (Date, Price)
  Todos los archivos:
     Fechas convertidas a datetime
     Ordenados cronológicamente (ascendente)
     Sin valores nulos
     Sin duplicados
     Consistencia verificada entre CSVs individuales y consolidado

Datasets resultantes:
------------------------------------------------------------
  df_x:       (9144, 2) — Materia prima X
  df_y:       (4485, 2) — Materia prima Y
  df_z:       (3565, 2) — Materia prima Z
  df_equipos: (3530, 6) — Histórico consolidado


## 8. Guardar datos procesados en S3

In [16]:
# Guardar cada dataset limpio
guardar_csv_s3(df_x, BUCKET, f'{PREFIX_PROCESADO}X_limpio.csv')
guardar_csv_s3(df_y, BUCKET, f'{PREFIX_PROCESADO}Y_limpio.csv')
guardar_csv_s3(df_z, BUCKET, f'{PREFIX_PROCESADO}Z_limpio.csv')
guardar_csv_s3(df_equipos, BUCKET, f'{PREFIX_PROCESADO}historico_equipos_limpio.csv')

print('\n Todos los datos procesados guardados en S3.')

Archivo guardado en s3://consulting-dataknow-prueba-tecnica/datos_procesados/X_limpio.csv
Archivo guardado en s3://consulting-dataknow-prueba-tecnica/datos_procesados/Y_limpio.csv
Archivo guardado en s3://consulting-dataknow-prueba-tecnica/datos_procesados/Z_limpio.csv
Archivo guardado en s3://consulting-dataknow-prueba-tecnica/datos_procesados/historico_equipos_limpio.csv

 Todos los datos procesados guardados en S3.


In [17]:
# Verificar que se guardaron correctamente
response = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX_PROCESADO)

print('Archivos en datos_procesados/:')
print('-' * 50)
for obj in response.get('Contents', []):
    if obj['Size'] > 0:
        print(f"  {obj['Key']:50s} {obj['Size']:>10,} bytes")

Archivos en datos_procesados/:
--------------------------------------------------
  datos_procesados/X_limpio.csv                         155,629 bytes
  datos_procesados/Y_limpio.csv                          77,442 bytes
  datos_procesados/Z_limpio.csv                          66,265 bytes
  datos_procesados/historico_equipos_limpio.csv         159,264 bytes
